# The Quality of the Question: AI Podcast Studio

**Description:**
A Python application that takes an excerpt from the entrepreneurship themed book, *The Quality of the Question*, and transforms it into a structured podcast episode.

**Repository:**
https://github.com/paolahsp/The-Quality-of-the-Question-AI-Podcast-Studio


**Notice:**
The source manuscript, *The Quality of the Question*, is © 2026 Paola Hintze. All rights reserved. This repository contains only an authorized excerpt for educational demonstration purposes. The original manuscript may not be reproduced or redistributed.

The Quality of the Question: AI Podcast Studio software developed by Paola Hintze, Marja Wiegand, and John Adams.

## Step 0: Pseudocode (Backend Development)

```
1. Setup environment
   - load required packages and libraries
   - load API key, initialize OpenAI client
   - validate loading success

2. Load and validate source text
   - read book excerpt from file
   - validate correct file types
   - validate not empty / not invalid

3. Define the JSON structure
   - Pydantic model for structured output
     (episode_title, opening_hook, book_introduction, book_journey,
     chapter_one_story, core_lesson, key_takeaways, reflection_question,
     listener_action, closing_teaser)
   - build system prompt
   - build user prompt with the source excerpt

4. LLM processing
   - call LLM, request structured JSON output
   - validate response against the Pydantic model

5. Output script for pass/fail review
   - present generated script to the user (frontend handles UI)
   - pass: continue to audio generation
   - fail: re-run step 4 to regenerate the script

6. Build narration text from the approved script
   - flatten the structured script fields into one narration string
   - validate the narration text is not empty

7. TTS processing
   - call TTS API with the narration text
   - validate audio was returned

8. Output files
   - save audio as MP3
   - save transcript (txt/json)
   - validate files were created / audio is playable

9. Placeholder download interface
   - only active once the output files exist
   - download button for audio, download button for transcript
   - real interface will be built by the frontend team
```

## Step 1: Import necessary packages and load instances

In [8]:
# Load libraries and connect to OpenAI API
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv

print("Loading environment...")

# Load API key from .env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "⚠️ The API key was not found in the .env file.\n"
        "How to fix: create a .env file in the repo root and set OPENAI_API_KEY=replace_with_your_key, then restart the kernel. "
    )

# Initialize client
client = OpenAI(api_key=api_key)

print("\n✅ Environment loaded successfully. OpenAI client initialized.")

Loading environment...

✅ Environment loaded successfully. OpenAI client initialized.


## Step 2: Load input from source

In [9]:
# Load and check the source text file
SOURCE_PATH = "data/source_texts/input_text_test_pooh.md"
ALLOWED_EXTENSIONS = (".txt", ".md")

print("Loading source text...")

# Validate file type before reading
if not SOURCE_PATH.lower().endswith(ALLOWED_EXTENSIONS):
    raise ValueError(
        f"⚠️ Unsupported file type for '{SOURCE_PATH}'. "
        f"Allowed types: {', '.join(ALLOWED_EXTENSIONS)}\n"
    )

# Read the source file
with open(SOURCE_PATH, "r", encoding="utf-8") as f:
    source_text = f.read().strip()

# Validate file is not empty
if not source_text:
    raise ValueError(f"⚠️ Source file '{SOURCE_PATH}' is empty. Add content and try again.")

word_count = len(source_text.split())

print(f"✅ Source loaded successfully: {SOURCE_PATH}")
print('\n' + '=' * 50)
print('OUTPUT')
print('=' * 50)
print(f"Word count: {word_count}")
print(f"Preview: {source_text[:200]}...")

Loading source text...
✅ Source loaded successfully: data/source_texts/input_text_test_pooh.md

OUTPUT
Word count: 2170
Preview: Here is Edward Bear, coming downstairs now, bump, bump, bump, on the back of his head, behind Christopher Robin. It is, as far as he knows, the only way of coming downstairs, but sometimes he feels th...


## Step 3: Define the JSON structure

In [10]:
# Set up the script schema and system prompt
from pydantic import BaseModel

# Structured output schema
class PodcastScript(BaseModel):
    episode_title: str
    opening_hook: str
    book_introduction: str
    book_journey: str
    chapter_one_story: str
    core_lesson: str
    key_takeaways: list[str]
    reflection_question: str
    listener_action: str
    closing_teaser: str

SYSTEM_PROMPT = """You are an expert podcast script writer adapting book excerpts into spoken-word episodes.

Rules you must follow:
- Use only information contained in the provided source excerpt.
- Preserve the author's central argument and meaning.
- Do not invent quotations, statistics, personal experiences, business cases, or biographical information.
- Do not copy the source excerpt word for word.
- Adapt the material into a natural spoken podcast format.
- Use one narrator.
- Maintain a thoughtful, intelligent, direct, and human tone.
- Include an opening hook, key takeaways, one reflection question, and one practical action.
- Return the result using the required structured format.
"""

print("✅ JSON structure and prompts defined")

✅ JSON structure and prompts defined


## Step 4: LLM processing

In [11]:
# Send the text to the LLM and get the script back
print("Transforming source text with LLM...")

try:
    completion = client.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": source_text}
        ],
        response_format=PodcastScript,
        # Add in a max token spend for testing, can be removed after validation
        max_tokens=1500,
    )
except Exception as e:
    raise RuntimeError(f"⚠️ LLM request failed: {e}. Check connection and API key, then retry.")

# Validate the response parsed successfully against the Pydantic model
podcast_script = completion.choices[0].message.parsed

if podcast_script is None:
    raise ValueError("⚠️ Model did not return a valid structured response. Try again.")

print("✅ Script generated and validated successfully.")
print('\n' + '=' * 50)
print('OUTPUT')
print('=' * 50)
print(podcast_script.model_dump_json(indent=2))

Transforming source text with LLM...
✅ Script generated and validated successfully.

OUTPUT
{
  "episode_title": "The Adventures of Winnie-the-Pooh",
  "opening_hook": "Imagine a bear who loves honey so much, he hatches clever plans to get it, even if those plans take him high up trees or lead to unexpected mishaps. Welcome to the whimsical world of Winnie-the-Pooh.",
  "book_introduction": "Today, we dive into the delightful universe created by A.A. Milne, where honey, friendship, and little adventures come to life. Join me as we explore the adventures of Winnie-the-Pooh—a bear whose curiosity often gets the better of him, leading to comical situations and heartwarming moments.",
  "book_journey": "In this episode, we follow Pooh as he embarks on a quest for honey, showcasing his ingenuity and the delightful dynamics he shares with his best friend, Christopher Robin. Through Pooh's bumbling escapades, we’ll highlight his mischief and resilience, reminding us that sometimes, it's the j

## Step 5: Output script for pass/fail review

In [12]:
# Show the script to the user for pass/fail review; on fail, regenerate with a note to try a different approach
approved = False
RETRY_NOTE = "\n\nNote: a previous attempt at this script was rejected. Try a different angle, opening hook, or structure this time."

while not approved:
    print('\n' + '=' * 50)
    print('SCRIPT FOR REVIEW')
    print('=' * 50)
    print(podcast_script.model_dump_json(indent=2))

    decision = input("\nPass or fail this script? Please type only Pass or Fail, then press enter. ").strip().lower()

    if decision == "pass":
        approved = True
        print("\n✅ Script approved. Continuing to audio generation.")
    elif decision == "fail":
        print("⚠️ Script failed. Regenerating with a different approach...")
        completion = client.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": source_text + RETRY_NOTE}
            ],
            response_format=PodcastScript,
            max_tokens=1500,
        )
        podcast_script = completion.choices[0].message.parsed
    else:
        print("Please select 'pass' or 'fail' to continue.")


SCRIPT FOR REVIEW
{
  "episode_title": "The Adventures of Winnie-the-Pooh",
  "opening_hook": "Imagine a bear who loves honey so much, he hatches clever plans to get it, even if those plans take him high up trees or lead to unexpected mishaps. Welcome to the whimsical world of Winnie-the-Pooh.",
  "book_introduction": "Today, we dive into the delightful universe created by A.A. Milne, where honey, friendship, and little adventures come to life. Join me as we explore the adventures of Winnie-the-Pooh—a bear whose curiosity often gets the better of him, leading to comical situations and heartwarming moments.",
  "book_journey": "In this episode, we follow Pooh as he embarks on a quest for honey, showcasing his ingenuity and the delightful dynamics he shares with his best friend, Christopher Robin. Through Pooh's bumbling escapades, we’ll highlight his mischief and resilience, reminding us that sometimes, it's the journey that teaches us the most.",
  "chapter_one_story": "Our story begi

## Step 6: Build narration text from the approved script

In [13]:
# Flatten the approved script's fields into one narration string for TTS input
print("Building narration text...")

# Flatten the structured script into one narration string, in spoken order
key_takeaways_text = " ".join(podcast_script.key_takeaways)

narration_text = " ".join([
    podcast_script.opening_hook,
    podcast_script.book_introduction,
    podcast_script.book_journey,
    podcast_script.chapter_one_story,
    podcast_script.core_lesson,
    key_takeaways_text,
    podcast_script.reflection_question,
    podcast_script.listener_action,
    podcast_script.closing_teaser,
])

# Validate the narration text isn't empty
if not narration_text.strip():
    raise ValueError("⚠️ Narration text is empty. Check the approved script.")

print(f"✅ Narration text built successfully.")
print('\n' + '=' * 50)
print('OUTPUT')
print('=' * 50)
print(f"Character count: {len(narration_text)}")
print(f"Preview: {narration_text[:200]}...")

Building narration text...
✅ Narration text built successfully.

OUTPUT
Character count: 2262
Preview: Imagine a bear who loves honey so much, he hatches clever plans to get it, even if those plans take him high up trees or lead to unexpected mishaps. Welcome to the whimsical world of Winnie-the-Pooh. ...


## Step 7: TTS processing

In [14]:
# Call OpenAI TTS with the narration text and validate audio was returned
TTS_VOICE = "nova"

print("Generating audio with TTS...")

try:
    tts_response = client.audio.speech.create(
        model="tts-1",
        voice=TTS_VOICE,
        input=narration_text,
    )
except Exception as e:
    raise RuntimeError(f"⚠️ TTS request failed: {e}. Check connection and API key, then retry.")

# Validate audio bytes were returned
audio_bytes = tts_response.content

if not audio_bytes:
    raise ValueError("⚠️ TTS did not return audio data. Try again.")

print("✅ Audio generated successfully.")
print('\n' + '=' * 50)
print('OUTPUT')
print('=' * 50)
print(f"Voice: {TTS_VOICE}")
print(f"Audio size: {len(audio_bytes)} bytes")

Generating audio with TTS...
✅ Audio generated successfully.

OUTPUT
Voice: nova
Audio size: 2813280 bytes


## Step 8: Save and validate output files

In [15]:
# Save the audio and transcript to the outputs folder, then validate both files exist
import json

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUDIO_PATH = os.path.join(OUTPUT_DIR, "episode.mp3")
TRANSCRIPT_PATH = os.path.join(OUTPUT_DIR, "transcript.json")

print("Saving output files...")

# Save audio
with open(AUDIO_PATH, "wb") as f:
    f.write(audio_bytes)

# Save transcript
with open(TRANSCRIPT_PATH, "w", encoding="utf-8") as f:
    json.dump(podcast_script.model_dump(), f, indent=2, ensure_ascii=False)

# Validate both files were created and are non-empty
if not os.path.exists(AUDIO_PATH) or os.path.getsize(AUDIO_PATH) == 0:
    raise ValueError(f"⚠️ Audio file was not created or is empty: {AUDIO_PATH}")

if not os.path.exists(TRANSCRIPT_PATH) or os.path.getsize(TRANSCRIPT_PATH) == 0:
    raise ValueError(f"⚠️ Transcript file was not created or is empty: {TRANSCRIPT_PATH}")

print("✅ Files saved and validated successfully.")
print('\n' + '=' * 50)
print('OUTPUT')
print('=' * 50)
print(f"Audio: {AUDIO_PATH} ({os.path.getsize(AUDIO_PATH)} bytes)")
print(f"Transcript: {TRANSCRIPT_PATH} ({os.path.getsize(TRANSCRIPT_PATH)} bytes)")

Saving output files...
✅ Files saved and validated successfully.

OUTPUT
Audio: outputs/episode.mp3 (2813280 bytes)
Transcript: outputs/transcript.json (2562 bytes)


## Step 9: Download audio or transcript

In [ ]:
# Simple placeholder UI to test downloads; the real interface is built by the frontend team
import gradio as gr

# Only enable the buttons once both output files exist
files_ready = os.path.exists(AUDIO_PATH) and os.path.exists(TRANSCRIPT_PATH)

if not files_ready:
    print("⚠️ Output files not found. Run steps 6-8 first.")
else:
    with gr.Blocks() as demo:
        gr.Markdown("## Podcast Studio — Download Placeholder")
        gr.DownloadButton("Download audio", value=AUDIO_PATH)
        gr.DownloadButton("Download transcript", value=TRANSCRIPT_PATH)

    demo.launch()